### Imports

In [1]:
import json
import warnings
from pathlib import Path

import torch
import torch.nn as nn

from src.config import CONFIG
from src.datasets.audio_dataset import AudioDataset
from src.engine import benchmark_snn, validate_snn, train_one_epoch_snn, get_split_dataloaders
from src.models.snn import SNN
from src.preprocessing import get_snn_pipeline
from src.utils import plot_training_history, run_sweep_pareto

warnings.filterwarnings("ignore", category=UserWarning)

C:\Users\jackr\Developer\bio-inspired-sonar-for-underwater-object-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Constants

In [2]:
MODEL_NAME = SNN.NAME

INPUT_DIR = Path('../../data/audio-mnist')
HYPERPARAMETERS_PATH = Path(f'../../hyperparameters/{MODEL_NAME}.json')
MODEL_PATH = Path(f'../../models/{MODEL_NAME}.pth')

NUM_EPOCHS = 20

### Device

In [3]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Using device: {device}')

Using device: cuda


### Hyperparameter Tuning

In [4]:
def objective(trial) -> tuple[float, int]:
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    beta = trial.suggest_float('beta_init', 0.5, 0.99)
    slope = trial.suggest_int('slope', 10, 50)
    timesteps = trial.suggest_categorical('timesteps', [5, 10, 15])

    dataset = AudioDataset(INPUT_DIR, get_snn_pipeline())
    train_dataloader, val_dataloader, _ = get_split_dataloaders(dataset)

    model = SNN(beta_init=beta, slope=slope, timesteps=timesteps).to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    val_acc = 0.0
    for epoch in range(CONFIG.hyperparameter_tuning.epochs):
        train_one_epoch_snn(device, model, criterion, optimiser, train_dataloader, leave=False)
        val_loss, val_acc = validate_snn(device, model, criterion, val_dataloader, leave=False)

    return val_acc, timesteps

if CONFIG.hyperparameter_tuning.should_run:
    run_sweep_pareto(objective, HYPERPARAMETERS_PATH)

[I 2026-03-11 22:14:51,641] A new study created in memory with name: no-name-e736e548-af9d-4dc1-b982-8d4f3498e02b


Running hyperparameter sweep (Pareto front)...


[I 2026-03-11 22:17:07,113] Trial 0 finished with values: [94.4, 5.0] and parameters: {'lr': 0.008812717717331619, 'beta_init': 0.6451259543736028, 'slope': 15, 'timesteps': 5}.
[I 2026-03-11 22:19:26,983] Trial 1 finished with values: [97.43333333333334, 15.0] and parameters: {'lr': 0.004530011179335062, 'beta_init': 0.940077758186522, 'slope': 15, 'timesteps': 15}.
[I 2026-03-11 22:21:38,450] Trial 2 finished with values: [96.86666666666666, 15.0] and parameters: {'lr': 0.000628499576494253, 'beta_init': 0.599155376471005, 'slope': 50, 'timesteps': 15}.
[I 2026-03-11 22:23:49,567] Trial 3 finished with values: [50.03333333333333, 15.0] and parameters: {'lr': 0.004239430455042939, 'beta_init': 0.9222049272991668, 'slope': 35, 'timesteps': 15}.
[I 2026-03-11 22:25:50,002] Trial 4 finished with values: [85.93333333333334, 5.0] and parameters: {'lr': 0.00014598711916038795, 'beta_init': 0.6742715473051711, 'slope': 27, 'timesteps': 5}.
[I 2026-03-11 22:27:54,128] Trial 5 finished with va

[REMEMBER TO MANUALLY SELECT BEST SET OF HYPERPARAMETERS]
Hyperparameter sweep completed.


### Training

In [ ]:
if __name__ == '__main__':
    dataset = AudioDataset(INPUT_DIR, get_snn_pipeline())
    train_dataloader, val_dataloader, test_dataloader = get_split_dataloaders(dataset)

    # Get one batch from the training loader and make sure it looks good
    features, labels = next(iter(train_dataloader))
    print(f'Features shape: {features.shape}')
    print(f'Labels shape: {labels.shape}')
    print()

    # Load hyperparameters
    # NOTE: Make sure to change which set of parameters to use, by default take the lowest timestep
    hyperparameters = json.load(open(HYPERPARAMETERS_PATH, 'r'))[1]['params']
    print(f'Hyperparameters used: {hyperparameters}')
    print()

    # Train the model
    model = SNN(beta_init=hyperparameters['beta_init'], slope=hyperparameters['slope'], timesteps=hyperparameters['timesteps']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=hyperparameters['lr'])
    criterion = nn.CrossEntropyLoss()

    print(f'Training {model.NAME}...')
    best_acc = 0.0
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    for epoch in range(NUM_EPOCHS):
        print(f'[Epoch {epoch + 1}/{NUM_EPOCHS}]')
        train_loss, train_acc = train_one_epoch_snn(device, model, criterion, optimizer, train_dataloader)
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        val_loss, val_acc = validate_snn(device, model, criterion, val_dataloader)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), MODEL_PATH)

        print(f'Train Loss: {train_loss:.2f} | Train Accuracy: {train_acc:.2f}% | Val Loss: {val_loss:.2f} | Val Accuracy: {val_acc:.2f}%')
        print()
    print(f'Best model had an accuracy of {best_acc:.2f}%.')
    print(f'Running final test...', end='')
    model.load_state_dict(torch.load(MODEL_PATH))

    test_accuracy, acs, first_layer_macs = benchmark_snn(device, model, test_dataloader, direct_encoded=True)
    print(f'Test Accuracy: {test_accuracy:.2f}% | Average ACs per Inference: {acs} | First Layer MACs: {first_layer_macs}')

    plot_training_history(train_losses, train_accs, val_losses, val_accs)

Features shape: torch.Size([64, 1, 64, 27])
Labels shape: torch.Size([64])

Hyperparameters used: {'lr': 0.0006623971484981565, 'beta_init': 0.9608107067021643, 'slope': 10, 'timesteps': 10}

Training snn_2d_direct...
[Epoch 1/20]


Validating: 100%|██████████| 47/47 [00:11<00:00,  4.03batches/s]


Train Loss: 0.97 | Train Accuracy: 68.91% | Val Loss: 0.31 | Val Accuracy: 90.40%

[Epoch 2/20]


Validating: 100%|██████████| 47/47 [00:04<00:00, 10.25batches/s]


Train Loss: 0.19 | Train Accuracy: 94.41% | Val Loss: 0.16 | Val Accuracy: 95.47%

[Epoch 3/20]


Validating:  62%|██████▏   | 29/47 [00:03<00:01, 11.79batches/s]